# Order table and baseline

Collapse order lines to one row per order, label it `any_return`, and fit an untuned LightGBM on the raw order attributes. This is the starting line later feature blocks are measured against.

**Config in one place.** `CUTOFF` is the time-split boundary. `gbm()` defines the single fixed, untuned LightGBM reused by every notebook in the project — the study varies the *data*, never the hyperparameters, so any score change is attributable to features.

In [1]:
import sys; sys.path.append("..")
import warnings; warnings.filterwarnings("ignore")
import json
import pandas as pd
import lightgbm as lgb
from sklearn.metrics import roc_auc_score, log_loss
from src.data import INTERIM, load_orders, save_features
from src.models import time_split

CUTOFF = "2015-07-01"
RAW_CATS = ["deviceID", "paymentMethod"]

def gbm():
    return lgb.LGBMClassifier(n_estimators=800, learning_rate=0.03, num_leaves=31, min_child_samples=50,
                              subsample=0.8, subsample_freq=1, colsample_bytree=0.7, reg_lambda=2.0,
                              n_jobs=-1, verbose=-1)

## Build

**The most consequential cell in the project: lines → orders.** `any_return = max(returned)` asks *did anything in this order come back*, because that is the unit a coverage policy pays out on. This is why the positive rate climbs from 52% at line level to 63.5% at order level: a five-line order has five chances to trigger.

In [2]:
lines = load_orders("orders_train")
lines["returned"] = (lines.returnQuantity > 0).astype(int)
lines["line_value"] = lines.price * lines.quantity

orders = lines.groupby("orderID").agg(
    orderDate=("orderDate", "min"),
    customerID=("customerID", "first"),
    deviceID=("deviceID", "first"),
    paymentMethod=("paymentMethod", "first"),
    n_lines=("orderID", "size"),
    n_items=("quantity", "sum"),
    order_value=("line_value", "sum"),
    voucher_amount=("voucherAmount", "max"),
    rrp_total=("rrp", "sum"),
    any_return=("returned", "max"),
).reset_index()
orders.shape

(738698, 11)

**Calendar features and persist.** `save_features` scrubs column names, downcasts to float32 and writes parquet. Every feature notebook ends this way, so all blocks share one on-disk contract.

In [3]:
orders["month"] = orders.orderDate.dt.month
orders["dow"] = orders.orderDate.dt.dayofweek
save_features(orders, "orders", keys=("orderID", "orderDate", "customerID") + tuple(RAW_CATS), target="any_return")
orders.head()

,orderID,orderDate,customerID,deviceID,paymentMethod,n_lines,n_items,order_value,voucher_amount,rrp_total,any_return,month,dow
0,a1000001,2014-01-01,c1010575,2,BPRG,2,2,30.00,0.0,69.98,0,1,2
1,a1000002,2014-01-01,c1045905,4,BPRG,2,2,84.99,0.0,99.98,1,1,2
2,a1000003,2014-01-01,c1089295,2,PAYPALVC,5,4,60.00,0.0,211.95,0,1,2
3,a1000004,2014-01-01,c1050116,3,BPRG,1,1,89.99,0.0,89.99,1,1,2
4,a1000005,2014-01-01,c1089296,2,BPRG,3,3,35.00,0.0,99.97,1,1,2


## Baseline

**Baseline feature set: raw order attributes only.** No bracketing, no history, no article rates. Categoricals are cast to `category` so LightGBM can split on category subsets natively instead of needing one-hot.

In [4]:
FEATS = ["n_lines", "n_items", "order_value", "voucher_amount", "rrp_total", "month", "dow"] + RAW_CATS
X = orders[FEATS].copy()
for c in RAW_CATS:
    X[c] = X[c].astype("category")
y = orders.any_return
tr, va = time_split(orders, "orderDate", CUTOFF)
tr.sum(), va.sum(), y[va].mean().round(4)

(np.int64(624732), np.int64(113966), np.float64(0.6304))

**Fit and score the baseline.** AUC 0.7959. This is the bar every later feature block has to clear, and the reason the engineered lift can be quoted honestly.

In [5]:
m = gbm().fit(X[tr], y[tr])
pred = m.predict_proba(X[va])[:, 1]
base = {"auc": roc_auc_score(y[va], pred), "logloss": log_loss(y[va], pred),
        "base_rate": float(y[tr].mean()), "valid_rate": float(y[va].mean())}
print(f"AUC {base['auc']:.4f}   logloss {base['logloss']:.4f}   valid base rate {base['valid_rate']:.4f}")

AUC 0.7959   logloss 0.5218   valid base rate 0.6304


## Save

**Persist the baseline.** Written to `data/interim/baseline.json` so notebook 07 can quote the before/after gap without re-running this one. Every notebook that needs a number from another reads it from disk rather than assuming execution order, which keeps each one independently runnable.

In [6]:
(INTERIM / "baseline.json").write_text(json.dumps(base, indent=2)); base

{'auc': 0.7958589260717418,
 'logloss': 0.521768459698951,
 'base_rate': 0.6359703040663837,
 'valid_rate': 0.6303897653686187}